##Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col


In [0]:
df=spark.table("workspace.bronze.erp_cust_az12")

In [0]:
for field in df.schema.fields:
    if field.dataType==StringType():
        df=df.withColumn(field.name,trim(col(field.name)))

In [0]:

df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)


##date validation

In [0]:


df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)


##Gender normalisation

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)



##Renaming

In [0]:
Rename={
    "cid":"customer_id"
    ,"bdate":"birth_date"
    ,"gen":"gender"
}
for old,new in Rename.items():
    df=df.withColumnRenamed(old,new)

In [0]:
df.display()

#Loading into silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.cust_az12")

#santiy check

In [0]:
%sql
select * from silver.cust_az12
